# 2026-11-14 MegaDepth-1500 held-out evaluation

Runs the 15 pre-registered configurations on MegaDepth-1500 (pose AUC). **Nothing is fitted on MegaDepth.**

**Before running:** upload this whole folder to `MyDrive/Colab Notebooks/2026-11-14_megadepth1500/`.

**Runtime:** CPU (Runtime -> Change runtime type -> CPU). No GPU is needed.

**Two steps, in this order:**
1. **Step 1 (pipeline check, float only).** Runs only C0 (float descriptors). In setting B its AUC should be close to the XFeat paper's 42.6 / 56.4 / 67.7 (AUC@5/10/20). About 30-40 minutes.
2. **Step 2 (the registered run).** Only after the check passes and `paper/preregistration_megadepth1500.md` is committed to git. Paste the commit hash into `PREREG_COMMIT`. About 3 hours. If Colab disconnects, run the cells again: finished configurations are skipped.

Results are saved in `results_colab/` inside this folder on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
EXP=Path('/content/drive/MyDrive/Colab Notebooks/2026-11-14_megadepth1500')
for p in ['md_eval.py','md_configs.py','md_pose.py','md_predictions.py','frozen_configs.npz','inputs/megadepth_1500.json','inputs/orig_code/features.py']:
    assert (EXP/p).exists(),p
(EXP/'results_colab').mkdir(exist_ok=True)
print('inputs found')

In [ ]:
import subprocess,sys,importlib.util,hashlib,tarfile
for module,package in {'cv2':'opencv-python-headless','torch':'torch','poselib':'poselib','gdown':'gdown','pandas':'pandas'}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
assert hashlib.sha256((EXP/'frozen_configs.npz').read_bytes()).hexdigest().startswith('8666ae9e8db35dab'), 'frozen_configs.npz differs from the registered file'
assert hashlib.sha256((EXP/'inputs/megadepth_1500.json').read_bytes()).hexdigest().startswith('6aa01108aa26b1d5'), 'pair list differs from the registered file'
# MegaDepth-1500 test images (same Google Drive file as XFeat's modules/dataset/download.py)
DATA=Path('/content/mega')
if not (DATA/'megadepth_test_1500').exists():
    import gdown
    DATA.mkdir(exist_ok=True)
    gdown.download('https://drive.google.com/uc?id=12yKniNWebDHRTCwhBNJmxYMPgqYX3Nhv', str(DATA/'test_images.tar'), quiet=False)
    with tarfile.open(DATA/'test_images.tar','r:tar') as tar: tar.extractall(path=DATA)
    (DATA/'test_images.tar').unlink()
print('images:', sum(1 for _ in (DATA/'megadepth_test_1500').rglob('*.jpg')))
# code runs from local disk; results go straight to Drive so that a disconnect loses nothing
subprocess.run(['rm','-rf','/content/run'],check=True)
subprocess.run(['cp','-r',str(EXP),'/content/run'],check=True)
ARGS=f'/content/mega /content/run/inputs/megadepth_1500.json /content/run/inputs/orig_code /content/work /content/run/frozen_configs.npz "{EXP}/results_colab"'
print(subprocess.run('lscpu | grep -E "Model name|^CPU\\(s\\)"',shell=True,capture_output=True,text=True).stdout)

## Step 1: pipeline check (float descriptors only)
Compare the `B / C0` row with the published XFeat numbers: **AUC@5 42.6, AUC@10 56.4, AUC@20 67.7**. A difference of a point or two is expected (RANSAC is random). A large gap means a pipeline problem: stop and report it.

In [ ]:
get_ipython().system(f'python /content/run/md_eval.py {ARGS} B,A C0 check')

## Step 2: the registered run
Fill in the git commit hash of the pre-registration first. Do not change anything else after seeing results.

In [ ]:
PREREG_COMMIT = ''   # <- paste the commit hash here
assert len(PREREG_COMMIT) >= 7, 'commit the pre-registration file first and paste its hash'
get_ipython().system(f'python /content/run/md_eval.py {ARGS} A,B all {PREREG_COMMIT}')

In [ ]:
get_ipython().system(f'python /content/run/md_predictions.py "{EXP}/results_colab"')
print(subprocess.run(['ls','-la',str(EXP/'results_colab')],capture_output=True,text=True).stdout[-2500:])